# Exploratory Data Analysis — RadarIQ Corrosion Dataset

This notebook explores the IQ radar signals for corrosion classification:
- Class distribution
- Signal visualisation
- Spectrograms
- Basic statistics

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline

from src.data_loading import load_or_generate_synthetic, validate_samples
from src.spectrograms import stft_spectrogram, compare_spectrograms
from src.preprocessing import extract_tabular_features
from src.utils import set_seed

set_seed(42)
print('Imports OK')

## 1. Load Synthetic Data

In [ ]:
data = load_or_generate_synthetic(
    n_classes=4,
    n_samples_per_class=200,
    signal_length=512,
    seed=42,
)

class_names = list(data.keys())
print('Classes:', class_names)
for name, arr in data.items():
    print(f'  {name}: {arr.shape}')

## 2. Class Distribution

In [ ]:
counts = {name: len(arr) for name, arr in data.items()}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(counts.keys(), counts.values(), color=['#4CAF50', '#2196F3', '#FF9800', '#F44336'])
ax.set_ylabel('Number of Samples')
ax.set_title('Class Distribution')
for bar, (name, cnt) in zip(bars, counts.items()):
    ax.text(bar.get_x() + bar.get_width()/2, cnt + 1, str(cnt),
            ha='center', va='bottom', fontsize=10)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()
print('Total samples:', sum(counts.values()))

## 3. Visualise Raw IQ Signals

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 6))
colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']

for ax, (name, arr), color in zip(axes.flat, data.items(), colors):
    t = np.linspace(0, 1, arr.shape[1])
    for i in range(min(5, len(arr))):
        ax.plot(t, arr[i], alpha=0.5, lw=0.8, color=color)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Time (normalised)')
    ax.set_ylabel('Amplitude')
    ax.grid(alpha=0.3)

fig.suptitle('Raw IQ Signals (5 examples per class)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Signal Statistics per Class

In [ ]:
import pandas as pd

rows = []
for name, arr in data.items():
    rows.append({
        'class': name,
        'mean': arr.mean(),
        'std': arr.std(),
        'min': arr.min(),
        'max': arr.max(),
        'rms': np.sqrt((arr**2).mean()),
    })

stats_df = pd.DataFrame(rows).set_index('class')
print(stats_df.round(4).to_string())

## 5. STFT Spectrograms

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
colors_cmap = ['Greens', 'Blues', 'Oranges', 'Reds']

for col, (name, arr) in enumerate(data.items()):
    # Raw signal
    axes[0, col].plot(arr[0], lw=0.8)
    axes[0, col].set_title(f'{name}\n(signal)', fontsize=8)
    axes[0, col].set_xlabel('Sample')

    # Spectrogram
    spec = stft_spectrogram(arr[0], n_fft=128, hop_length=32)
    im = axes[1, col].imshow(spec, aspect='auto', origin='lower',
                              cmap=colors_cmap[col])
    axes[1, col].set_title(f'{name}\n(spectrogram)', fontsize=8)
    axes[1, col].set_xlabel('Time frame')
    axes[1, col].set_ylabel('Freq bin')
    plt.colorbar(im, ax=axes[1, col])

fig.suptitle('IQ Signals and STFT Spectrograms', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Tabular Feature Distributions

In [ ]:
feature_names = [
    'mean', 'std', 'skewness', 'kurtosis', 'ptp', 'rms',
    'spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff',
    'peak1', 'peak2', 'peak3',
]

all_feats = {}
for name, arr in data.items():
    feats = np.stack([extract_tabular_features(s) for s in arr])
    all_feats[name] = feats

# Plot feature distributions for a few key features
plot_feat_idxs = [0, 1, 6, 7]  # mean, std, centroid, bandwidth
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
c_list = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']

for ax, fi in zip(axes, plot_feat_idxs):
    for (name, feats), color in zip(all_feats.items(), c_list):
        ax.hist(feats[:, fi], bins=20, alpha=0.6, label=name, color=color)
    ax.set_title(feature_names[fi], fontsize=10)
    ax.set_xlabel('Value')
    ax.legend(fontsize=6)

fig.suptitle('Tabular Feature Distributions per Class', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Validation Report

Check the synthetic data passes validation.

In [ ]:
valid, report = validate_samples(data)
print('Validation Report:')
for cls, rep in report.items():
    print(f'  {cls}: {rep}')